In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [2]:
load_dotenv()

True

In [3]:
model = ChatOpenAI()

In [4]:
# State 
class JokeState(TypedDict):
    topic: str
    joke: str
    explanation: str

In [5]:
# Function For generate Joke
def generate_joke(state: JokeState) -> JokeState:
    prompt = f'Generate a joke about {state["topic"]}.'
    response = model.invoke(prompt).content
    
    return {'joke': response}

In [6]:
# Function for Explanation Joke
def explain_joke(state: JokeState) -> JokeState:
    prompt = f'Explain the joke: {state["joke"]}.'
    response = model.invoke(prompt).content
    
    return {'explanation': response}
    

In [7]:
# Graph 
graph = StateGraph(JokeState)

# Nodes
graph.add_node("Generate_Joke", generate_joke)
graph.add_node("Explain_Joke", explain_joke)

# Edges
graph.add_edge(START, "Generate_Joke")
graph.add_edge("Generate_Joke", "Explain_Joke")
graph.add_edge("Explain_Joke", END)

# Checkpointer
checkpointer = InMemorySaver()

# Compile 
workflow = graph.compile(checkpointer=checkpointer)

In [8]:
# Config
config1 = {"configurable": {"thread_id": "1"}}

# Execute
initial_state = {"topic": "Python"}

final_state = workflow.invoke(initial_state, config=config1)
final_state

{'topic': 'Python',
 'joke': "Why did the python programmer go broke?\nBecause he couldn't C# his way out of a paper bag!",
 'explanation': 'This joke plays on the fact that Python and C# are both programming languages. The punchline suggests that the python programmer went broke because he was unable to "C# his way out of a paper bag" - meaning he was unable to use the C# programming language to solve his financial problems. It\'s a play on the literal meaning of the phrase "can\'t X your way out of a paper bag," which is used to indicate that someone is incompetent or lacking in a certain skill.'}

In [9]:
# Thred id
workflow.get_state(config1)

StateSnapshot(values={'topic': 'Python', 'joke': "Why did the python programmer go broke?\nBecause he couldn't C# his way out of a paper bag!", 'explanation': 'This joke plays on the fact that Python and C# are both programming languages. The punchline suggests that the python programmer went broke because he was unable to "C# his way out of a paper bag" - meaning he was unable to use the C# programming language to solve his financial problems. It\'s a play on the literal meaning of the phrase "can\'t X your way out of a paper bag," which is used to indicate that someone is incompetent or lacking in a certain skill.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0746ee-f579-62f8-8002-70f382a47290'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2025-08-08T15:47:05.920362+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0746ee-e302-6091-8001-1524725e45d9'}}, tasks=(),

In [10]:
# Intermediate State
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'Python', 'joke': "Why did the python programmer go broke?\nBecause he couldn't C# his way out of a paper bag!", 'explanation': 'This joke plays on the fact that Python and C# are both programming languages. The punchline suggests that the python programmer went broke because he was unable to "C# his way out of a paper bag" - meaning he was unable to use the C# programming language to solve his financial problems. It\'s a play on the literal meaning of the phrase "can\'t X your way out of a paper bag," which is used to indicate that someone is incompetent or lacking in a certain skill.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0746ee-f579-62f8-8002-70f382a47290'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2025-08-08T15:47:05.920362+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0746ee-e302-6091-8001-1524725e45d9'}}, tasks=()

In [11]:
# Config
config2 = {"configurable": {"thread_id": "2"}}

# Execute
initial_state = {"topic": "pizza"}

final_state = workflow.invoke(initial_state, config=config2)
final_state

{'topic': 'pizza',
 'joke': 'Why did the pizza go to the therapist?\n\nBecause it wanted to get a little slice of life advice!',
 'explanation': 'This joke is a play on words, as it combines the literal meaning of a pizza "slice" with the figurative meaning of getting advice or insights on life. The joke suggests that the pizza went to the therapist to get some guidance on life, but also utilizes a pun by referring to the pizza slice.'}

In [12]:
# Thread ID
workflow.get_state(config2)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the therapist?\n\nBecause it wanted to get a little slice of life advice!', 'explanation': 'This joke is a play on words, as it combines the literal meaning of a pizza "slice" with the figurative meaning of getting advice or insights on life. The joke suggests that the pizza went to the therapist to get some guidance on life, but also utilizes a pun by referring to the pizza slice.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f0746ef-3277-6d66-8002-a3a7c54a18a6'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2025-08-08T15:47:12.316120+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f0746ef-2553-6f47-8001-ce3ba85380cb'}}, tasks=(), interrupts=())

In [13]:
# Intermediate State
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the therapist?\n\nBecause it wanted to get a little slice of life advice!', 'explanation': 'This joke is a play on words, as it combines the literal meaning of a pizza "slice" with the figurative meaning of getting advice or insights on life. The joke suggests that the pizza went to the therapist to get some guidance on life, but also utilizes a pun by referring to the pizza slice.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f0746ef-3277-6d66-8002-a3a7c54a18a6'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2025-08-08T15:47:12.316120+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f0746ef-2553-6f47-8001-ce3ba85380cb'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the therapist?\n\nBecause it wanted to get a little slice of life advice!'

# Time Travel

In [14]:
workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f0746ee-d9be-6c2d-8000-e66b1bc55856"}})

StateSnapshot(values={'topic': 'Python'}, next=('Generate_Joke',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f0746ee-d9be-6c2d-8000-e66b1bc55856'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2025-08-08T15:47:03.012855+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0746ee-d9ba-6dea-bfff-a3d8d8816802'}}, tasks=(PregelTask(id='d90a00ad-ba92-5396-af68-f89136f37ea3', name='Generate_Joke', path=('__pregel_pull', 'Generate_Joke'), error=None, interrupts=(), state=None, result={'joke': "Why did the python programmer go broke?\nBecause he couldn't C# his way out of a paper bag!"}),), interrupts=())

In [15]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f0746ee-d9be-6c2d-8000-e66b1bc55856"}})

{'topic': 'Python',
 'joke': 'Why was the python so good at programming? \n\nBecause it always slithered through code with ease!',
 'explanation': 'This joke plays on the dual meaning of "python" - it can refer to the programming language, Python, or to the actual snake, python. The joke suggests that the python (snake) is good at programming (writing code) because it is able to effortlessly "slither" through the code, implying that it navigates it easily and efficiently.'}

In [16]:
# Intermediate State
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'Python', 'joke': 'Why was the python so good at programming? \n\nBecause it always slithered through code with ease!', 'explanation': 'This joke plays on the dual meaning of "python" - it can refer to the programming language, Python, or to the actual snake, python. The joke suggests that the python (snake) is good at programming (writing code) because it is able to effortlessly "slither" through the code, implying that it navigates it easily and efficiently.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0746fd-0245-6806-8002-7b531f1bd8f4'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2025-08-08T15:53:23.071970+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0746fc-f692-63b6-8001-ac9cf50a4c54'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'Python', 'joke': 'Why was the python so good at programming? \n\nBecause it a

# Update State

In [17]:
workflow.update_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f0746ee-d9be-6c2d-8000-e66b1bc55856", "checkpoint_ns": ""}}, {'topic':'samosa'})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f074705-97e1-6524-8001-136df6ec2f5b'}}

In [18]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa'}, next=('Generate_Joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f074705-97e1-6524-8001-136df6ec2f5b'}}, metadata={'source': 'update', 'step': 1, 'parents': {}}, created_at='2025-08-08T15:57:13.507955+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0746ee-d9be-6c2d-8000-e66b1bc55856'}}, tasks=(PregelTask(id='bfa0da82-20d8-4463-a667-8079cb85cb20', name='Generate_Joke', path=('__pregel_pull', 'Generate_Joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'Python', 'joke': 'Why was the python so good at programming? \n\nBecause it always slithered through code with ease!', 'explanation': 'This joke plays on the dual meaning of "python" - it can refer to the programming language, Python, or to the actual snake, python. The joke suggests that the python (snake) is good at programming (writing code) 

In [20]:
workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f074705-97e1-6524-8001-136df6ec2f5b"}})

StateSnapshot(values={'topic': 'samosa'}, next=('Generate_Joke',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f074705-97e1-6524-8001-136df6ec2f5b'}}, metadata={'source': 'update', 'step': 1, 'parents': {}}, created_at='2025-08-08T15:57:13.507955+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0746ee-d9be-6c2d-8000-e66b1bc55856'}}, tasks=(PregelTask(id='bfa0da82-20d8-4463-a667-8079cb85cb20', name='Generate_Joke', path=('__pregel_pull', 'Generate_Joke'), error=None, interrupts=(), state=None, result=None),), interrupts=())

In [21]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f074705-97e1-6524-8001-136df6ec2f5b"}})

{'topic': 'samosa',
 'joke': 'Why did the samosa go to the party? Because it heard there would be a lot of "filling"!',
 'explanation': 'This joke plays on the double meaning of the word "filling." In the context of a samosa, "filling" refers to the delicious mixture inside the fried pastry. However, in the context of a party, "filling" can refer to the large number of people attending. So, the joke is saying that the samosa went to the party because it heard there would be a lot of people (or things) to fill it up, like a party-goer might be filled up with food.'}